# Load

In [190]:
import pandas as pd
from pathlib import Path
import re

In [191]:
# base = Path(r"d:\Ace\tata_micro\quantin")

# Load customer purchase behavior data
df_purchases = pd.read_csv("../data/QVI_purchase_behaviour.csv")

# Load transaction data
df_transactions = pd.read_excel("../data/QVI_transaction_data.xlsx", sheet_name=0)

# See

In [192]:
df_purchases.head()

,LYLTY_CARD_NBR,LIFESTAGE,PREMIUM_CUSTOMER
0,1000,YOUNG SINGLES/COUPLES,Premium
1,1002,YOUNG SINGLES/COUPLES,Mainstream
2,1003,YOUNG FAMILIES,Budget
3,1004,OLDER SINGLES/COUPLES,Mainstream
4,1005,MIDAGE SINGLES/COUPLES,Mainstream


In [193]:
df_transactions.head()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
0,43390,1,1000,1,5,Natural Chip Compny SeaSalt175g,2,6.0
1,43599,1,1307,348,66,CCs Nacho Cheese 175g,3,6.3
2,43605,1,1343,383,61,Smiths Crinkle Cut Chips Chicken 170g,2,2.9
3,43329,2,2373,974,69,Smiths Chip Thinly S/Cream&Onion 175g,5,15.0
4,43330,2,2426,1038,108,Kettle Tortilla ChpsHny&Jlpno Chili 150g,3,13.8


In [194]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264836 entries, 0 to 264835
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   DATE            264836 non-null  int64  
 1   STORE_NBR       264836 non-null  int64  
 2   LYLTY_CARD_NBR  264836 non-null  int64  
 3   TXN_ID          264836 non-null  int64  
 4   PROD_NBR        264836 non-null  int64  
 5   PROD_NAME       264836 non-null  object 
 6   PROD_QTY        264836 non-null  int64  
 7   TOT_SALES       264836 non-null  float64
dtypes: float64(1), int64(6), object(1)
memory usage: 16.2+ MB


In [195]:
df_transactions['PROD_QTY'].describe()

count    264836.000000
mean          1.907309
std           0.643654
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max         200.000000
Name: PROD_QTY, dtype: float64

In [196]:

print(df_transactions[["PROD_NAME", "PROD_QTY"]].sort_values(by="PROD_QTY", ascending=False))

                                       PROD_NAME  PROD_QTY
69762           Dorito Corn Chp     Supreme 380g       200
69763           Dorito Corn Chp     Supreme 380g       200
217237              Pringles Sweet&Spcy BBQ 134g         5
238333            Pringles SourCream  Onion 134g         5
238471   Infuzions BBQ Rib   Prawn Crackers 110g         5
...                                          ...       ...
82354     Cobs Popd Sour Crm  &Chives Chips 110g         1
82357                   CCs Tasty Cheese    175g         1
172438                        Kettle Chilli 175g         1
82358      Grain Waves Sour    Cream&Chives 210G         1
32479   Smiths Crinkle Cut  French OnionDip 150g         1

[264836 rows x 2 columns]


In [197]:
# drop rows with the outlier prod quantity
df_transactions = df_transactions[df_transactions["PROD_QTY"] != 200]

In [198]:
df_transactions.nunique()

DATE                 364
STORE_NBR            272
LYLTY_CARD_NBR     72636
TXN_ID            263125
PROD_NBR             114
PROD_NAME            114
PROD_QTY               5
TOT_SALES            111
dtype: int64

In [199]:
df_transactions.sample(5)

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
119890,43552,32,32085,28524,96,WW Original Stacked Chips 160g,2,3.8
32012,43516,269,269004,264984,21,WW Sour Cream &OnionStacked Chips 160g,2,3.8
32712,43603,212,212185,211496,42,Doritos Corn Chip Mexican Jalapeno 150g,2,7.8
36989,43388,57,57233,52634,31,Infzns Crn Crnchers Tangy Gcamole 110g,2,7.6
123230,43498,81,81384,81428,51,Doritos Mexicana 170g,2,8.8


In [200]:
df_transactions.nunique()

DATE                 364
STORE_NBR            272
LYLTY_CARD_NBR     72636
TXN_ID            263125
PROD_NBR             114
PROD_NAME            114
PROD_QTY               5
TOT_SALES            111
dtype: int64

In [201]:
df_purchases.nunique()

LYLTY_CARD_NBR      72637
LIFESTAGE               7
PREMIUM_CUSTOMER        3
dtype: int64

In [202]:
df_transactions.duplicated().sum() 

1

In [203]:
for x in ['LIFESTAGE', 'PREMIUM_CUSTOMER']:
    # print(f"\n{x}")
    print(df_purchases[x].value_counts())
    print("-------------------------------\n")

LIFESTAGE
RETIREES                  14805
OLDER SINGLES/COUPLES     14609
YOUNG SINGLES/COUPLES     14441
OLDER FAMILIES             9780
YOUNG FAMILIES             9178
MIDAGE SINGLES/COUPLES     7275
NEW FAMILIES               2549
Name: count, dtype: int64
-------------------------------

PREMIUM_CUSTOMER
Mainstream    29245
Budget        24470
Premium       18922
Name: count, dtype: int64
-------------------------------



In [204]:
# ending in G
# pattern = r".*(G)$"

# ending in 3digits and G
pattern = r".*\b\d{1,4}\s*(G)$"


df_transactions["ends_with_weight"] = df_transactions["PROD_NAME"].str.match(pattern, na=False)

print("Names that end with weight:")
print(df_transactions[df_transactions["ends_with_weight"]][["PROD_NBR", "PROD_NAME"]].drop_duplicates())

print("\nNames that do NOT end with weight:")
print(df_transactions[~df_transactions["ends_with_weight"]][["PROD_NBR", "PROD_NAME"]].drop_duplicates())

Names that end with weight:
    PROD_NBR                                PROD_NAME
9         52    Grain Waves Sour    Cream&Chives 210G
34        48  Red Rock Deli Sp    Salt & Truffle 150G

Names that do NOT end with weight:
     PROD_NBR                                 PROD_NAME
0           5    Natural Chip        Compny SeaSalt175g
1          66                  CCs Nacho Cheese    175g
2          61    Smiths Crinkle Cut  Chips Chicken 170g
3          69    Smiths Chip Thinly  S/Cream&Onion 175g
4         108  Kettle Tortilla ChpsHny&Jlpno Chili 150g
..        ...                                       ...
520        58     Red Rock Deli Chikn&Garlic Aioli 150g
521        10       RRD SR Slow Rst     Pork Belly 150g
528        11                  RRD Pc Sea Salt     165g
549        43        Smith Crinkle Cut   Bolognese 150g
689        41                  Doritos Salsa Mild  300g

[112 rows x 2 columns]


In [205]:
df_transactions.loc[df_transactions["PROD_NAME"] == "KETTLE 135G SWT POT SEA SALT", "PROD_NAME"] = "KETTLE SWT POT SEA SALT 135G"

# Change1

In [206]:
df_transactions['DATE'] = pd.to_datetime(
    df_transactions['DATE'],
    unit='D',
    origin='1899-12-30'
)

In [207]:
# make all uppercase remove trailing spaces convert to string
df_transactions["PROD_NAME"] = df_transactions["PROD_NAME"].astype(str).str.strip().str.upper()

# remove double spaces in product names
df_transactions["PROD_NAME"] = (df_transactions["PROD_NAME"].str.split().str.join(" "))

In [208]:
# extract packet_weight from product name

df_transactions["PACKET_WEIGHT"] = (
    df_transactions["PROD_NAME"]
    .str.extract(r'(\d+)\s*G\s*$', expand=False)
    .astype("Int64")
)


In [209]:
df_transactions["PROD_NAME"] = (
    df_transactions["PROD_NAME"]
    .str.replace(r'(\d+)\s*G\s*$', '', regex=True)
    .str.strip()
)

In [210]:
df_transactions[["PROD_NAME", "PACKET_WEIGHT"]].head()

,PROD_NAME,PACKET_WEIGHT
0,NATURAL CHIP COMPNY SEASALT,175
1,CCS NACHO CHEESE,175
2,SMITHS CRINKLE CUT CHIPS CHICKEN,170
3,SMITHS CHIP THINLY S/CREAM&ONION,175
4,KETTLE TORTILLA CHPSHNY&JLPNO CHILI,150


In [211]:
# 1. Check for products where weight extraction failed
print(df_transactions[df_transactions['PACKET_WEIGHT'].isna()]['PROD_NAME'].unique())


['KETTLE 135G SWT POT SEA SALT']


In [212]:
# 3. Look at unique product names to scan for non-chip items and brand patterns
sorted(df_transactions['PROD_NAME'].unique())



['BURGER RINGS',
 'CCS NACHO CHEESE',
 'CCS ORIGINAL',
 'CCS TASTY CHEESE',
 'CHEETOS CHS & BACON BALLS',
 'CHEETOS PUFFS',
 'CHEEZELS CHEESE',
 'CHEEZELS CHEESE BOX',
 'COBS POPD SEA SALT CHIPS',
 'COBS POPD SOUR CRM &CHIVES CHIPS',
 'COBS POPD SWT/CHLLI &SR/CREAM CHIPS',
 'DORITO CORN CHP SUPREME',
 'DORITOS CHEESE SUPREME',
 'DORITOS CORN CHIP MEXICAN JALAPENO',
 'DORITOS CORN CHIP SOUTHERN CHICKEN',
 'DORITOS CORN CHIPS CHEESE SUPREME',
 'DORITOS CORN CHIPS NACHO CHEESE',
 'DORITOS CORN CHIPS ORIGINAL',
 'DORITOS MEXICANA',
 'DORITOS SALSA MEDIUM',
 'DORITOS SALSA MILD',
 'FRENCH FRIES POTATO CHIPS',
 'GRAIN WAVES SOUR CREAM&CHIVES',
 'GRAIN WAVES SWEET CHILLI',
 'GRNWVES PLUS BTROOT & CHILLI JAM',
 'INFUZIONS BBQ RIB PRAWN CRACKERS',
 'INFUZIONS MANGO CHUTNY PAPADUMS',
 'INFUZIONS SOURCREAM&HERBS VEG STRWS',
 'INFUZIONS THAI SWEETCHILI POTATOMIX',
 'INFZNS CRN CRNCHERS TANGY GCAMOLE',
 'KETTLE 135G SWT POT SEA SALT',
 'KETTLE CHILLI',
 'KETTLE HONEY SOY CHICKEN',
 'KETTLE MOZZAREL

In [213]:
# 4. Check customer file for nulls / duplicate loyalty numbers
print(df_purchases.isna().sum())
print(df_purchases['LYLTY_CARD_NBR'].duplicated().sum())

LYLTY_CARD_NBR      0
LIFESTAGE           0
PREMIUM_CUSTOMER    0
dtype: int64
0


In [214]:
df_transactions['YEAR'] = df_transactions['DATE'].dt.year
df_transactions['MONTH'] = df_transactions['DATE'].dt.month
df_transactions['DAY'] = df_transactions['DATE'].dt.day
df_transactions['MONTH_NAME'] = df_transactions['DATE'].dt.month_name()
df_transactions['WEEKDAY'] = df_transactions['DATE'].dt.day_name()
df_transactions['QUARTER'] = df_transactions['DATE'].dt.quarter

In [215]:
# # since we are doing only chips
# # check and remove the products that are salsa and dip

# mask = df_transactions['PROD_NAME'].str.contains('SALSA|DIP', case=False, na=False)
# print(df_transactions.loc[mask, 'PROD_NAME'].unique())
# print(f"Rows affected: {mask.sum()}")

In [217]:
# df_transactions_salsadip = df_transactions[mask].reset_index(drop=True)

In [218]:
brand_map = {
    'SMITH': 'SMITHS',
    'SMITHS': 'SMITHS',
    'SNBTS': 'SUNBITES',
    'SUNBITES': 'SUNBITES',
    'DORITO': 'DORITOS',
    'DORITOS': 'DORITOS',
    'WW': 'WOOLWORTHS',
    'WOOLWORTHS': 'WOOLWORTHS',
    'NCC': 'NATURAL CHIP CO',
    'NATURAL': 'NATURAL CHIP CO',   # careful, see below
    'INFZNS': 'INFUZIONS',
    'INFUZIONS': 'INFUZIONS',
    'RRD': 'RED ROCK DELI',
    'RED': 'RED ROCK DELI',         # careful, see below
    'GRNWVES': 'GRAIN WAVES',
    'GRAIN': 'GRAIN WAVES',         # careful, see below
    'CCS': 'CCS',
    'COBS': 'COBS',
    'TYRRELLS': 'TYRRELLS',
    'THINS': 'THINS',
    'TWISTIES': 'TWISTIES',
    'CHEETOS': 'CHEETOS',
    'TOSTITOS': 'TOSTITOS',
    'KETTLE': 'KETTLE',
    'PRINGLES': 'PRINGLES',
}

In [219]:
df_transactions['BRAND'] = df_transactions['PROD_NAME'].str.split().str[0]
df_transactions['BRAND'] = df_transactions['BRAND'].replace(brand_map)

In [220]:
# exclude_products = [
#     'INFUZIONS BBQ RIB PRAWN CRACKERS',
#     'INFUZIONS MANGO CHUTNY PAPADUMS',
#     'INFUZIONS SOURCREAM&HERBS VEG STRWS'
# ]
# df_transactions = df_transactions[~df_transactions['PROD_NAME'].isin(exclude_products)].reset_index(drop=True)

In [221]:
# for name in ['CHEEZELS CHEESE', 'CHEEZELS CHEESE BOX']:
#     print(name, df_transactions.loc[df_transactions['PROD_NAME'] == name, 'PROD_NBR'].unique())

# for name in ['SMITHS CRINKLE CHIPS SALT & VINEGAR', 'SMITHS CRINKLE CUT SALT & VINEGAR']:
#     print(name, df_transactions.loc[df_transactions['PROD_NAME'] == name, 'PROD_NBR'].unique())

# for name in ['SMITHS CRINKLE CUT CHIPS ORIGINAL', 'SMITHS CRINKLE ORIGINAL']:
#     print(name, df_transactions.loc[df_transactions['PROD_NAME'] == name, 'PROD_NBR'].unique())

In [222]:
# sort longest (most words) first, so "NATURAL CHIP COMPNY" is matched
# before a shorter overlapping key could grab part of it
for raw, standardized in sorted(brand_map.items(), key=lambda x: -len(x[0].split())):
    pattern = r'^' + re.escape(raw) + r'\b'
    df_transactions['PROD_NAME'] = df_transactions['PROD_NAME'].str.replace(pattern, standardized, regex=True)

In [223]:
# Check if any PROD_NBR has more than one PROD_NAME
prod_nbr_to_names = df_transactions.groupby("PROD_NBR")["PROD_NAME"].nunique()
multi_name_prods = prod_nbr_to_names[prod_nbr_to_names > 1]

print("PROD_NBR with more than one PROD_NAME:")
print(multi_name_prods)

# Check if any PROD_NAME has more than one PROD_NBR
prod_name_to_numbers = df_transactions.groupby("PROD_NAME")["PROD_NBR"].nunique()
multi_number_prods = prod_name_to_numbers[prod_name_to_numbers > 1]

print("\nPROD_NAME with more than one PROD_NBR:")
print(multi_number_prods)

PROD_NBR with more than one PROD_NAME:
Series([], Name: PROD_NAME, dtype: int64)

PROD_NAME with more than one PROD_NBR:
Series([], Name: PROD_NBR, dtype: int64)


In [224]:
# For any PROD_NBR with multiple names, show exactly what those names are
if len(multi_name_prods) > 0:
    problem_nbrs = multi_name_prods.index
    print(df_transactions[df_transactions['PROD_NBR'].isin(problem_nbrs)][['PROD_NBR', 'PROD_NAME']].drop_duplicates().sort_values('PROD_NBR'))

# Same for the reverse case
if len(multi_number_prods) > 0:
    problem_names = multi_number_prods.index
    print(df_transactions[df_transactions['PROD_NAME'].isin(problem_names)][['PROD_NBR', 'PROD_NAME']].drop_duplicates().sort_values('PROD_NAME'))

In [225]:
# Exact duplicate rows across the whole transaction record
print(df_transactions.duplicated().sum())

# Same TXN_ID appearing more than once with different content is worth a look too
print(df_transactions['TXN_ID'].duplicated().sum())

1
1709


In [226]:
df_transactions = df_transactions.drop_duplicates().reset_index(drop=True)

In [227]:
df_transactions.to_csv('QVI_transaction_data_clean.csv', index=False)

In [228]:
txn_cards = set(df_transactions['LYLTY_CARD_NBR'].unique())
cust_cards = set(df_purchases['LYLTY_CARD_NBR'].unique())

print("Cards in transactions but not in customer file:", len(txn_cards - cust_cards))
print("Cards in customer file but not in transactions:", len(cust_cards - txn_cards))

Cards in transactions but not in customer file: 0
Cards in customer file but not in transactions: 1


In [229]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264833 entries, 0 to 264832
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   DATE              264833 non-null  datetime64[ns]
 1   STORE_NBR         264833 non-null  int64         
 2   LYLTY_CARD_NBR    264833 non-null  int64         
 3   TXN_ID            264833 non-null  int64         
 4   PROD_NBR          264833 non-null  int64         
 5   PROD_NAME         264833 non-null  object        
 6   PROD_QTY          264833 non-null  int64         
 7   TOT_SALES         264833 non-null  float64       
 8   ends_with_weight  264833 non-null  bool          
 9   PACKET_WEIGHT     261576 non-null  Int64         
 10  YEAR              264833 non-null  int32         
 11  MONTH             264833 non-null  int32         
 12  DAY               264833 non-null  int32         
 13  MONTH_NAME        264833 non-null  object        
 14  WEEK

In [230]:
df_transactions.to_csv('QVI_transaction_data_clean.csv', index=False)

In [231]:
df_merged = df_transactions.merge(df_purchases, on='LYLTY_CARD_NBR', how='left')

print(len(df_merged) == len(df_transactions))
print(df_merged.isna().sum())

True
DATE                   0
STORE_NBR              0
LYLTY_CARD_NBR         0
TXN_ID                 0
PROD_NBR               0
PROD_NAME              0
PROD_QTY               0
TOT_SALES              0
ends_with_weight       0
PACKET_WEIGHT       3257
YEAR                   0
MONTH                  0
DAY                    0
MONTH_NAME             0
WEEKDAY                0
QUARTER                0
BRAND                  0
LIFESTAGE              0
PREMIUM_CUSTOMER       0
dtype: int64


In [232]:
df_merged.to_csv('QVI_merged_data.csv', index=False)

In [233]:
print(df_transactions[df_transactions['PACKET_WEIGHT'].isna()]['PROD_NAME'].unique())

['KETTLE 135G SWT POT SEA SALT']
